# U-Net input ablations: Ddc / full Yb / imaginary Yb / real Yb

Updated interface for `dl_normalized_datasets` v2. Set CASE=30 or 57. The terminal runner and notebook use the same code. Run from this code folder.


In [1]:
from pathlib import Path
import subprocess
import sys
import json
import csv

CODE_DIR = Path.cwd()  # Folder containing train_ablation.py and updated model files
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 39              # 30, 39 or 57
TARGET = 'dq'          # 'dq' or 'vmag'; one target type per run
MODELS = ['unet', 'attention_unet']
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real']
SEEDS = [42, 123, 2026]
EPOCHS = 200
BATCH_SIZE = 64
LOSS_ALPHA = 0.5       # 1.0 means pure MSE
DEVICE = 'cuda'
SMOKE = False         # True uses tiny subsets/two epochs; not a performance estimate
RESUME = False
AMP = True

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}'
assert (CODE_DIR / 'train_ablation.py').is_file(), 'Set CODE_DIR to the extracted updated code folder'
for name in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{name}.npz').is_file(), f'Missing {name} data'
print('Data:', DATA_DIR)
print('Results:', OUT_DIR)


Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case39
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case39_dq


In [2]:
command = [sys.executable, '-u', str(CODE_DIR / 'train_ablation.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--models', *MODELS, '--variants', *VARIANTS,
    '--target', TARGET, '--seeds', *map(str, SEEDS),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--loss-alpha', str(LOSS_ALPHA), '--device', DEVICE]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
if not AMP: command.append('--no-amp')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/train_ablation.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case39 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case39_dq --models unet attention_unet --variants ddc ddc_yb ddc_imag ddc_real --target dq --seeds 42 123 2026 --epochs 200 --batch-size 64 --loss-alpha 0.5 --device cuda
PyTorch 2.10.0+cu128 device cuda
Dataset(dq, X=(6837, 3, 39, 39), y=(6837, 1, 39, 39)) 
 Dataset(dq, X=(1465, 3, 39, 39), y=(1465, 1, 39, 39))

unet ddc seed=42: 1,927,009 parameters; AMP=True
epoch=  1 loss=3.37812 val_R2=0.6844059661110308 val_RMSE=2.867190 (20.7s)
epoch=  2 loss=0.28451 val_R2=0.7724735251720061 val_RMSE=2.434490 (2.4s)
epoch=  3 loss=0.24203 val_R2=0.8110259922635724 val_RMSE=2.218672 (2.3s)
epoch=  4 loss=0.21279 val_R2=0.81118554738068 val_RMSE=2.217735 (2.4s)
epoch=  5 loss=0.19747 val_R2=0.8215

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/train_ablation.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case39', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case39_dq', '--models', 'unet', 'attention_unet', '--variants', 'ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', '--target', 'dq', '--seeds', '42', '123', '2026', '--epochs', '200', '--batch-size', '64', '--loss-alpha', '0.5', '--device', 'cuda'], returncode=0)

In [3]:
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Metrics use original units:', summary['protocol']['metric_units'])
for model, variants in summary['aggregate'].items():
    for variant, splits in variants.items():
        print(model, variant, 'test:', splits['test'])
print('Training-mean baseline:', summary['training_mean_baseline'])
print('Individual runs:', OUT_DIR / 'comparison.csv')


Metrics use original units: raw DQ
unet ddc test: {'r2': {'mean': 0.8630634538331199, 'std_across_seeds': 0.0010094179795783682}, 'rmse': {'mean': 1.8887178944833913, 'std_across_seeds': 0.006954921903432235}, 'mae': {'mean': 0.7618206222316096, 'std_across_seeds': 0.009402475292436334}}
unet ddc_yb test: {'r2': {'mean': 0.8592368523195141, 'std_across_seeds': 0.0005118969475756761}, 'rmse': {'mean': 1.9149320983627554, 'std_across_seeds': 0.0034834066735807156}, 'mae': {'mean': 0.8242371926203601, 'std_across_seeds': 0.011002565857934426}}
unet ddc_imag test: {'r2': {'mean': 0.8598463062055582, 'std_across_seeds': 0.0014411000378943821}, 'rmse': {'mean': 1.9107673481399159, 'std_across_seeds': 0.009835814900402334}, 'mae': {'mean': 0.811152205491983, 'std_across_seeds': 0.025765966118033137}}
unet ddc_real test: {'r2': {'mean': 0.8607969146730275, 'std_across_seeds': 0.0019704229835984066}, 'rmse': {'mean': 1.9042613108963014, 'std_across_seeds': 0.013485280154287382}, 'mae': {'mean':

## Notes

All four variants share the same fitted training normalization, parameter count within each architecture, initial weights per seed, and training protocol. Excluded slots are masked after normalization. Test data is evaluated only after validation checkpoint selection. Do not apply a second scaler or sigmoid to standardized targets. The saved predictions and R2/RMSE/MAE are in original units. See README.md for inference and recovery.
